# Pulpermill backend (Kaggle launcher)
Starts the Pulpermill model server on Kaggle and gives you a public address to type into the page.

**Before you run**
1. In the session options, set **Accelerator = GPU T4 x2** and **Internet = on**.
2. Under Add-ons > Secrets, add `PULPERMILL_TOKEN` (any long random string; you type the same one into the page) and `NGROK_AUTHTOKEN` (from your ngrok account), and make sure both are attached to this notebook. Optionally add `HF_TOKEN` (a Hugging Face read token): the model then downloads signed in, which is faster.
3. Optional: attach a dataset that holds the model file (Add Input). It is used as it is and nothing is downloaded.

**Then use Run All.** The last cell keeps running. When the server is ready it prints a box with the address: type that address into **Model connection** on the page, and your `PULPERMILL_TOKEN` into the token box. The address changes every session. The first run takes several minutes, longer if the model has to download.

**To stop,** stop the session. To start again, run all the cells from the top.

## Settings
The only cell you may want to change.

In [ ]:
BRANCH = "main"        # which version of the code to run: a branch or a release tag of the repository
PORT = 8000            # the port the server listens on (the tunnel points at it)
PROMPTER = "mill"      # the wording module: "mill" is the real one; "stub" answers with placeholders and is only for trying the page

# Diagnostics, for finding out why a call was refused. Leave both off for normal use.
# Both write story text to /kaggle/working: do not share or commit those files.
DEBUG = False          # every refused reply and its problem line, to pulpermill-debug.jsonl
TRACE = False          # every prompt and reply of every attempt, to pulpermill-trace.jsonl

## 1. Get the code and install
Clones the repository fresh (so the code that runs is the code on GitHub, not an old copy) and installs the server's packages.

In [ ]:
import os, shutil, subprocess, sys

REPO = "https://github.com/LacChe/pulpermill"
DIR = "/kaggle/working/pulpermill"

# The code was already imported in this session (an earlier run): a new clone would not replace it, so stop here.
if "pulpermill" in sys.modules:
    raise RuntimeError("The code was already imported in this session. Stop the session and run all the cells again, so the fresh clone is the one that runs.")

os.chdir("/kaggle/working")
shutil.rmtree(DIR, ignore_errors=True)         # a fresh clone every session: an old copy would keep running the old code
try:
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO, DIR], check=True)
except subprocess.CalledProcessError as error:
    raise RuntimeError(f"Could not clone {REPO} (branch or tag {BRANCH!r}). Check that Internet is on and that BRANCH names a real branch or tag.") from error
os.chdir(DIR)
sys.path.insert(0, DIR)
print("running commit", subprocess.check_output(["git", "rev-parse", "--short", "HEAD"]).decode().strip())

# The server's own packages. llama-cpp-python comes in step 3, with CUDA.
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "pulpermill/backend/requirements.txt"], check=True)

## 2. Check the session
Checks what would otherwise fail after a long wait: two GPUs, Internet, the model, the secrets. If anything is wrong it stops with a list of everything to fix, before the slow steps.

In [ ]:
from pulpermill.backend.kaggle_setup import check_session
secrets = check_session()      # PULPERMILL_TOKEN and NGROK_AUTHTOKEN are required; HF_TOKEN is optional

## 3. llama-cpp-python with CUDA
Loads the CUDA 12 runtime libraries (Kaggle's image may carry only CUDA 13), then uses the llama-cpp-python that is installed if it loads and can use the GPU, and otherwise installs a ready-made CUDA 12 build. It must be run again after a restart. The first run takes a few minutes.

In [ ]:
from pulpermill.backend.kaggle_setup import ensure_cuda_llama
ensure_cuda_llama()

## 4. Start the server
Loads the model, answers one short test sentence, starts the server and opens the public address. This cell keeps running; the address is printed when it is ready.

**Model file:** if an attached dataset holds `mlabonne_gemma-3-27b-it-abliterated-Q6_K.gguf`, it is used as it is. Otherwise it downloads to `/kaggle/temp/models`, which has far more room than `/kaggle/working`.

In [ ]:
from pulpermill.backend.launch import launch

launch(token=secrets["PULPERMILL_TOKEN"], ngrok_authtoken=secrets["NGROK_AUTHTOKEN"], hf_token=secrets["HF_TOKEN"],
       port=PORT, prompter=PROMPTER, debug=DEBUG, trace=TRACE)

## If something goes wrong
- **"The session is not ready"** (step 2): the list names each thing to fix: the accelerator, Internet, or a secret that is not set or not attached to this notebook. Fix them and run all the cells again.
- **"Not enough disk space for the model"**: attach a dataset that holds the model file, or free space in `/kaggle/temp`.
- **"cannot use the GPU"** (step 3): the accelerator is not GPU T4 x2.
- **The page says unauthorized**: the token in the page must be exactly your `PULPERMILL_TOKEN` secret.
- **The page cannot connect**: the address changes every session. Copy the new one from the box printed by the last cell.
- More in `docs/running-and-configuration.md` (Troubleshooting).